# Notebook 3 <br>Data quality and combining data

In Notebook 1 you learned to *look* at data and see what was wrong with it. In Notebook 2
you repaired a broken export. This notebook does two things: it makes the validating of data 
more explicit and it introduces the other common challange data processingL data you need is 
usually spread over more than one table.

Start by loading pandas:

In [ ]:
import pandas as pd
import numpy as np
import pooch

%reload_ext autoreload
%autoreload 2

pd.set_option("display.precision", 2)

This notebook builds all its data inside the notebook itself, so there is no data file to fetch. It does need `tests.py`, which checks your answers. Run the cell below to download it into the folder this notebook is in. You only need to do this once; after that the file is already on your computer and the cell finishes immediately.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/spcourse/pandas/main/pandas-dp/data/"

print("downloading data files")
for fname in ["tests.py"]:
    pooch.retrieve(url=DATA_URL + fname, known_hash=None, fname=fname, path=".", progressbar=True)
print("done!")

from tests import *

## Chapter 10: Validating data

Up to now, checking your data has been implicit: after every step you looked at `shape`,
`dtypes`, a few rows, `isna().sum()`. Let's make the types of questions you genrally would 
like to answer about the data a bit more explicit:

| question | tools for finding out |
|---|---|
| Is anything in here twice? | `duplicated`, `drop_duplicates` |
| Is the same thing always written the same way? | the `.str` accessor, `isin`, the `category` dtype |
| Does every value have the right *shape*? | regular expressions: `.str.fullmatch`, `.str.extract` |
| Is every value *plausible*? | domain rules, `between`, quantiles |

### Duplicates

A duplicate is a row that says something the data already said. They arrive constantly: an
export gets run twice, two systems both report the same event, someone submits a form
twice because the first attempt seemed slow.

Here is a small order list.

In [ ]:
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 102, 104, 105],
    "customer": ["Ada", "Ben", "Cleo", "Ben", "Dan", "Ada"],
    "item":     ["cable", "mouse", "screen", "mouse", "cable", "keyboard"],
    "amount":   [12.50, 25.00, 189.00, 25.00, 12.50, 45.00],
})

display(orders)
print("duplicated rows:", orders.duplicated().sum())

`duplicated()` returns a boolean Series, one value per row. You can use this to find the duplicates: 

In [ ]:
# keep=False marks *every* copy, including the first -- use it to look before you delete
display(orders[orders.duplicated(keep=False)])

And this is how you can remove them:

In [ ]:
deduped = orders.drop_duplicates()

print("rows:", len(orders), "->", len(deduped))
print("expected to drop:", orders.duplicated().sum())

`drop_duplicates()` will remove everything silently. So always inspect the duplicates first, count how many were removed in the end and check if this matches your expectations. As we did above.

This was relatively easy because the rows were identical, so it did not matter which copy
survived. It become harder when the same thing appears twice with different data attached.

This price list has one product listed twice.

In [ ]:
prices = pd.DataFrame({
    "sku":     ["X1", "X2", "X3", "X2"],
    "price":   [9.99, 14.50, 7.25, 15.95],
    "updated": ["2024-01-04", "2024-01-04", "2024-01-04", "2024-03-12"],
})

display(prices)
print("exact duplicate rows:", prices.duplicated().sum())
print("duplicate skus:      ", prices.duplicated(subset="sku").sum())

The *sku* (Stock Keeping Unit) is a unique code that shops assign to a specific product they sell, so we would like each sku to be unique in the data. Here we can directly see that they aren't, but with large DataFrames you can use `is_unique` on a column. And we can use `.duplicated` on a single column to see the specific duplications.

In [ ]:
print("sku is unique:", prices["sku"].is_unique)

display(prices[prices[["sku"]].duplicated(keep=False)])

We can use `subset="sku"` to specify it to look at only that column. But, which `X2` is right? Pandas cannot tell you. `9.99` and `15.95` are both plausible prices. The only thing that distinguishes them is that one was recorded later. 
Let's assume we have enough demain knowledge to know that later is better, we can use `keep="last"`to specify 
that we only keep the last occurance. 

In [ ]:
latest = (prices
          .sort_values("updated")
          .drop_duplicates(subset="sku", keep="last"))

display(latest)
print("sku is unique:", latest["sku"].is_unique)

### Exercise 21

Below is a sign-up list. It has both kinds of problem: one row was submitted twice, and one
person signed up again later on a different plan.

Build a DataFrame called `accounts` with one row per email address, keeping each
person's most recent sign-up.

Your result should have 4 rows, a unique `email` column, and (since you're keeping the last entry) you should have `ben@uva.nl` on the `pro` plan, not `free`.

In [ ]:
signups = pd.DataFrame({
    "email":     ["ada@uva.nl", "ben@uva.nl", "ada@uva.nl",
                  "cleo@uva.nl", "ben@uva.nl", "dan@uva.nl"],
    "plan":      ["free", "free", "free", "pro", "pro", "free"],
    "signed_up": ["2024-02-01", "2024-02-03", "2024-02-01",
                  "2024-02-05", "2024-03-09", "2024-03-11"],
})

accounts = None

# YOUR CODE HERE

print("rows:", len(signups), "->", len(accounts))
print("email is unique:", accounts["email"].is_unique)
display(accounts)

In [ ]:
test_21(accounts, signups)

### Checking the shape of a value

For textual collumns it can often be a bit more involved to validate all values. How do you know it something is indeed a postcodes, email addresses, or ISBN? 

A regular expression can be very useful here. It is a way to check if a string matches a specific pattern. For example, Dutch postcodes have four digits, a space and then two letters (e.g., "1017 CV"). The following regular expression captures that:

`r"\d{4} [A-Z]{2}"`

`\d{4}` means four digits, and `[A-Z]{2}` means 2 capital letters.

This is not a course on regular expressions, so you're really not required to know how to construct them. But you should know they exist and when they can be useful.

If you'd like to learn more anyway (it's also not that hard). This interactive tutorial explains it very well: [https://regexone.com/]

The example below shows how we can use regular expression with `fullmatch` to check which entries are valid postcodes: 

In [ ]:
codes = pd.Series(["1017 CV", "3511LN", "971 CP", "5611-EM", "2513 AAA", "6211 LK"])

pattern = r"\d{4} [A-Z]{2}"
display(pd.DataFrame({"code": codes, "valid": codes.str.fullmatch(pattern)}))

Only two of the six are valid. 

> The `r` in front of the pattern makes it a *raw string*, which stops Python from treating
the backslash in `\d` as an escape character. Always write patterns as raw strings.

Pay attention to the difference between `match` and `fullmatch`:

In [ ]:
display(pd.DataFrame({
    "code":      codes,
    "match":     codes.str.match(pattern),
    "fullmatch": codes.str.fullmatch(pattern),
}))

print("match says   ", codes.str.match(pattern).sum(), "codes are valid")
print("fullmatch says", codes.str.fullmatch(pattern).sum(), "codes are valid")

- `match` only looks at the start of the value: "does this value
*begin* with something matching?". So, "2513 AAA" macthes as it starts width a valid postcode ("2513 AA").
- `fullmatch` requires the entire value to match.

### Plausible values: ranges, rules and outliers

A value can be the right type, spelled consistently and perfectly formatted, and still be impossible.

Here is a small set of product reviews.

In [ ]:
reviews = pd.DataFrame({
    "review_id":     [1, 2, 3, 4, 5, 6, 7, 8],
    "rating":        [4.5, 3.0, 47.0, 5.0, 2.5, 4.0, -1.0, 3.5],
    "posted":        pd.to_datetime(["2024-03-01", "2024-02-11", "2024-01-30", "1900-01-01",
                                     "2024-03-14", "2024-02-02", "2024-03-05", "2024-12-31"]),
    "helpful_votes": [3, 0, 1, 2, 140, 0, 1, 2],
})

display(reviews.describe())

`describe()` is the smell test you already know, and it is doing real work here: a maximum
rating of 47 and a minimum of −1 are visible immediately, and the mean rating of 8.56 is
obviously nonsense. But `describe()` only *shows* you the numbers. It does not know that a rating runs from 1
to 5.

It is good practice to write down explicit tests that can validate your data after reading it.

In [ ]:
# create a data-health summary
flagged_rating = ~reviews["rating"].between(1, 5)
print(f"# rating outside 1-5: {flagged_rating.sum()}")

flagged_date = reviews["posted"] < "2024-01-01"
print(f"# posted before 2024: {flagged_date.sum()}")

This data-health summaryallowsa to read any issues at a glance. And every time you update your data you can quickly rerun the same tests again. It also helps you to see if modifactions (repairs) have any influence on your data health.

Of course, for this to be an effective tool, you'd add many more check to such a summary.

You don't alway want to drop bad rows. Review 4 has an impossible *date*, but its rating may be perfectly good, 
and throwing the row away loses that. It can be better to just flag them using the rules you create above:|

In [ ]:
flagged = pd.concat([flagged_rating, flagged_date], axis=1).any(axis=1)

reviews["suspect"] = flagged
print("rows flagged:", flagged.sum(), "of", len(reviews))
display(reviews)

The `pd.concat(..., axis=1)` syntax is still unfamiliar. It puts the three boolean masks side by side as three columns, and
`.any(axis=1)` asks, per row, whether any of them is `True`. You will see `concat` properly
in the next chapter.

#### Quantiles and the IQR

Sometimes nothing tells you what is plausible. Nobody can say how many "helpful" votes a
review is allowed to get. In that situation the only thing you can do is to look for outliers.

There is many ways tom compute outliers. But a simple way to do that is to use quartiles: 

Q1 and Q3 are the points a quarter and three quarters of the way through the sorted data. For example, 
sorted, the votes here are `0 0 1 1 2 2 3 140`, and Q1 (a quarter of the way along) lands
between the second `0` and the first `1`, so pandas interpolates between them and reports
`0.75`. Q3 lands between 2 and 3, and gets interprolated to `2.25`. 

The distance between Q1 and Q3 is the *interquartile range*, and anything more than
1.5 IQRs beyond either is conventionally called an outlier.

In [ ]:
votes = reviews["helpful_votes"]

q1, q3 = votes.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 {q1}, Q3 {q3}, IQR {iqr}")
print(f"plausible range: {low} to {high}")
print("outliers:", votes[(votes < low) | (votes > high)].tolist())

It finds the 140. Be careful with what that means. The IQR rule is a heuristic, we do often not know if it is actually wrong.

This is important to keep in mind: with a domain rule (`rating` is between 1 and 5) we know for sure a 
certain value cannot exist, an outlier only signlas we should have a good look at it.

### Exercise 22

Write a data-health summary for the shipments below, in the same style as the one above.
Three rules apply:

- `weight_kg` must be greater than 0 and at most 100
- `days_in_transit` must be between 1 and 30, inclusive
- `country` must be one of `ALLOWED_COUNTRIES`

For each rule, make a boolean mask marking the rows that are **wrong** and print one line
saying how many there are. Call the masks `flagged_weight`, `flagged_transit` and
`flagged_country`.

Then combine the three into `flagged`, which is `True` for every row breaking at least one
rule. **4** of the 8 shipments are clean.

In [ ]:
shipments = pd.DataFrame({
    "shipment":        [1, 2, 3, 4, 5, 6, 7, 8],
    "weight_kg":       [2.4, 0.0, 15.2, 8.8, 3.1, 120.0, 6.7, 0.9],
    "days_in_transit": [3, 2, 45, 4, 1, 2, 5, 3],
    "country":         ["NL", "BE", "NL", "DE", "XX", "NL", "FR", "BE"],
})

ALLOWED_COUNTRIES = ["NL", "BE", "DE", "FR"]

flagged_weight = None
flagged_transit = None
flagged_country = None
flagged = None

# YOUR CODE HERE

print("\nrows flagged:", flagged.sum(), "of", len(shipments))
display(shipments[flagged])

In [ ]:
test_22(flagged_weight, flagged_transit, flagged_country, flagged, shipments)

### Conclusion

Having a good validation discipline is vital when reading data. You should alway distrust the sanity of 
the data you're reading.

But, there is another reason to care about all of this: everything so far has assumed a single table. 
As soon as you start combining multiple tables, it becomes very easy to introduce errors that you will 
not notice unless you look for them.

## Chapter 11: Combining data

Everything so far has assumed one table. Real data almost never arrives that way: the track
list is in one file, the artists in another, last month's numbers in a third.

This chapter is about putting them together. There are realy only two things you can do to combine data: match
rows up by a shared value, or stack them on top of each other. But those operations come in different variations.

> **Video.** The first half of this chapter follows the order of explanation in
> [Merging DataFrames in Pandas](https://www.youtube.com/watch?v=TPivN7tpdwc&list=PLUaB-1hjhk8GZOuylZqLz-Qt9RIdZZMBE&index=5)
> by Alex The Analyst. It is a good companion to the sections below, in this order:
> `merge`, then `join`, then `concat`.
>
> One warning: the video finishes with `df.append()`. That method was removed in pandas 2
> and no longer exists — see the `concat` section.

### Merging on a key

Two tables, both about the same music platform.

In [ ]:
artists = pd.DataFrame({
    "artist_id": [1, 2, 3, 4],
    "name":      ["Neon Harbour", "Kite Season", "Vera Lune", "Odd Even"],
    "country":   ["NL", "BE", "FR", "NL"],
})

tracks = pd.DataFrame({
    "track_id":  [10, 11, 12, 13, 14],
    "title":     ["Slow Tide", "Harbour Lights", "First Frost", "Paper Boats", "Night Bus"],
    "artist_id": [1, 1, 2, 3, 9],
    "streams":   [48_000, 125_000, 9_000, 61_000, 3_000],
})

display(artists)
display(tracks)

The two share a column: `artist_id`. That shared column is the *key* (the thing Pandas uses to see what belongs togehter).

`merge` matches rows on the key (`on="..."`)

In [ ]:
merged = pd.merge(tracks, artists, on="artist_id")

display(merged)
print("tracks:", len(tracks), " artists:", len(artists), " merged:", len(merged))

When we merge this way, you see we wind up with four rows. Even though the `tracks` DataFrame contained out of five tracks. Something was dropped. By default `merge` keeps only rows that matched on both sides. This is called an *inner* join. Here
two things fail to match:

- `Night Bus` has `artist_id` 9, and there is no artist 9.
- `Odd Even` (artist 4) has no tracks.

Neither appears in the result. This is the single most common way a merge goes wrong, and
it is invisible unless you look for it: the output is a perfectly ordinary DataFrame that is
simply missing things.

In [ ]:
# Which keys did not survive?
print("track artist_ids without an artist:",
      sorted(set(tracks["artist_id"]) - set(artists["artist_id"])))
print("artists without a track:          ",
      sorted(set(artists["artist_id"]) - set(tracks["artist_id"])))

If you do an inner join it is always good to check which elements you lost this way.

### Keeping what doesn't match: `how=`

You can also keep all the elements from both tables (also the ones that don't match) by doing an *outer* join (`how="outer"`):
`how=` decides.

In [ ]:
merged_outer = pd.merge(tracks, artists, on="artist_id", how="outer")
display(merged_outer)

So in this case, no information is lost. Every row that could not be matched it padded with `NaN` values. For example For artist with id `4` we had a name (`Odd Even`), but no track. So all track columns (`track_id`, `title`, `streams`) are filled with `NaN`. Similarly, for artist with id `9`, we do have track info, but no name or country.

You can also use a `left` join if you want to keep all rows from the left DataFrame (and drop the unmatchin ones from the right). And a `right` join for the oposit.

In [ ]:
display(pd.merge(tracks, artists, on="artist_id", how="left"))

`Night Bus` survives, with an empty name and country. A

Note that `left` and `right` are not about importance, only about which frame you
wrote first. So, `pd.merge(a, b, how="left")` and `pd.merge(b, a, how="right")` give the same
rows.

### Exercise 23

Using the same `tracks` and `artists` create a DataFrame `artist_info` that combines `track` info and `artist` info, but only keeps the rows if we have full information (name, country) on the artist. Make sure that *all* artist in `artists` show up in the resulting `artist_info` DataFrame, also if we don't have track info about them.

In [ ]:
artist_info = None

# YOUR CODE HERE

print("catalogue:", len(artist_info), "rows")
display(artist_info)

In [ ]:
test_23(artist_info, tracks, artists)

### Every combination: `how="cross"`

There is a fifth option, and it does not use a key at all. A *cross join* pairs every row
on the left with every row on the right.

In [ ]:
combinations = pd.merge(artists, tracks, how="cross")

print(len(artists), "artists x", len(tracks), "tracks =", len(combinations), "rows")
display(combinations)

This yields every combination of every row of the first DataFrame with every row from the second. This is also called a *carthesian product* and lays at the basis of how databases operate on tables. It contains 20 rows: 4 (from `artists`) $\times$ 5 (from `tracks`). 

Note that column names that occur in both tables get suffixed (`_x` and `_y`) to tell them appart: `artist_id_x` and `artist_id_y`.

Be careful with this merge: it can make your data explode: two frames of 10 000 rows produce
a hundred million rows. So it's smart to always check the size before you run one, with
`len(left) * len(right)`.

So why does it exist? Because sometimes "every possible combination" is exactly the list
you need. We'll come back to this later.

One thing to note for now is that we can quite easily reproduce an innerjoin using the `"cross"`:

In [ ]:
## only keep rows where artist_id_x equals artist_id_y

alternative_inner_join = combinations[combinations["artist_id_x"] == combinations["artist_id_y"]]
display(alternative_inner_join)

### `join`: merging on the index

`join` does the same work as `merge`, but it matches on the index rather than on a
column. If the thing you want to match on is already the index, it is shorter.

In [ ]:
artists_by_id = artists.set_index("artist_id")
display(artists_by_id)

# match tracks' artist_id column against artists' index
display(tracks.join(artists_by_id, on="artist_id"))

Or, with both frames indexed by the key, no `on=` is needed at all:

In [ ]:
display(tracks.set_index("artist_id").join(artists_by_id))

Attention! `join` defaults to `how="left"`, while `merge` defaults to `how="inner"`. Swapping
one for the other therefore changes your row count even when everything else looks
identical. Both accept `how=`, so say what you mean.

One more difference: if both frames have a column with the same name, `merge` quietly
renames them to `name_x` and `name_y`, but `join` refuses and raises
`ValueError: columns overlap but no suffix specified`. Give it `lsuffix` and `rsuffix`:

In [ ]:
left  = pd.DataFrame({"name": ["Slow Tide", "First Frost"]}, index=[1, 2])
right = pd.DataFrame({"name": ["Neon Harbour", "Kite Season"]}, index=[1, 2])

display(left.join(right, lsuffix="_track", rsuffix="_artist"))

### Stacking: `concat`

So, `merge` and `join` widen your data same rows, more columns. `concat` makes it longer: same columns, more rows. You use it when data is for example split by time or by source, like one export per month.

In [ ]:
export_a = pd.DataFrame({
    "track_id": [10, 11, 10, 11],
    "month":    ["Jan", "Jan", "Feb", "Feb"],
    "streams":  [400, 300, 450, 280],
})

export_b = pd.DataFrame({
    "track_id": [10, 11, 10, 11],
    "month":    ["Feb", "Feb", "Mar", "Mar"],
    "streams":  [450, 280, 500, 310],
})

stacked = pd.concat([export_a, export_b], ignore_index=True)
display(stacked)

`ignore_index=True` renumbers the result. Without it you keep both original indexes and end
up with two rows numbered 0, two numbered 1, and so on (this breaks `.loc` and create a mess later on).

Now look at the data. The two exports overlap: February is in both. This is a common problem that we already pointed out in Chapter 10.

In [ ]:
monthly = stacked.drop_duplicates()

print("stacked:", len(stacked), "-> deduplicated:", len(monthly))
print("expected: 2 tracks x 3 months =", 2 * 3)

### Exercise 24

Consider the three weekly exports of the same report below. They have some overlap: the weeks on the boundary between exports boundary weeks appear twice.

Combine them into `plays`, with no duplicated rows and a clean index.

Before you work out the answer, work out what the row count *should* be: how many tracks are
there, and how many distinct weeks? Print that expectation first.

Tip: you will need to look at the `ignore_index=True` argument of `contcat` and the `reset_index(drop=True)` method.

In [ ]:
week_1 = pd.DataFrame({"track_id": [10, 11], "week": [1, 1], "plays": [120, 80]})
week_2 = pd.DataFrame({"track_id": [10, 11, 10, 11], "week": [1, 1, 2, 2],
                       "plays": [120, 80, 140, 95]})
week_3 = pd.DataFrame({"track_id": [10, 11, 10, 11], "week": [2, 2, 3, 3],
                       "plays": [140, 95, 160, 70]})

plays = None

# YOUR CODE HERE

print("expected number of rows: ?")
print("actual number of rows:", len(plays))

display(plays)

In [ ]:
test_24(plays, week_1, week_2, week_3)

### What goes wrong: cardinality

The dangerous merges are not the ones that lose rows. They are the ones that *add* them.

Before merging, you should think about how the two frames relate:

- **one-to-one**: each key appears once on both sides. Row count stays the same.
- **one-to-many**: each key appears once on one side and several times on the other. This
  is the case we've been looking at: one artist, many tracks. So the key (`artist_id` is unique in `artists`, but not in `tracks`).
- **many-to-many**: the key repeats on both sides. Every combination is produced, and the
  row count multiplies.

The third is often an accident. For example when there is a duplicate artist in our data:

In [ ]:
artists_dup = pd.DataFrame({
    "artist_id": [1, 1, 2, 3, 4],
    "name":      ["Neon Harbour", "Neon Harbour", "Kite Season", "Vera Lune", "Odd Even"],
    "country":   ["NL", "NL", "BE", "FR", "NL"],
})

careless = pd.merge(tracks, artists_dup, on="artist_id", how="left")

print("tracks:", len(tracks), "-> merged:", len(careless))
display(careless)

Five tracks became seven rows. `Slow Tide` and `Harbour Lights` each matched *both* copies
of artist 1, so each is now listed twice. (This is why Chapter 10 spent a section on duplicate keys. `artists_dup["artist_id"].is_unique`)

`merge` can also check it for you, which is better because it cannot be forgotten:

In [ ]:
pd.merge(artists_dup, tracks, on="artist_id", how="left", validate="one_to_many")

`validate=` takes `"one_to_one"`, `"one_to_many"`, `"many_to_one"` or `"many_to_many"` and
raises a `MergeError` if the data does not match your claim. Writing down what you *expect*
turns a silent corruption into a loud stop.

The other tool is `indicator=True`, which adds a `_merge` column recording where each row
came from. It answers "what didn't match?" without any set arithmetic:

In [ ]:
audited = pd.merge(tracks, artists, on="artist_id", how="outer", indicator=True)

display(audited["_merge"].value_counts())
display(audited)

Four rows matched, one track has no artist, one artist has no track.

### Exercise 25

Each artist belongs to exactly one record label, so merging `artists` with `labels` should
give 4 rows (one per artist), but it doesn't.

Find out why, fix it, and produce `with_labels` with the 4 rows it should have.

Add `validate="one_to_one"` to your merge. If your fix is right the merge runs; if it isn't,
pandas will give you an error. (Why should it be one-to-one in this case?)

In [ ]:
labels = pd.DataFrame({
    "artist_id": [1, 2, 3, 3, 4],
    "label":     ["Tidal", "Northwind", "Halcyon", "Halcyon", "Tidal"],
})

with_labels = None

# YOUR CODE HERE

print("rows:", len(with_labels))
display(with_labels)

In [ ]:
test_25(with_labels, artists, labels)

### When can `how="cross"` be useful?

This si a bit of a sidequest, but now it would be good to have a look again at `merge(..., how="cross")`, as promised before.

Here is an operation that produces results that might seem suprising (and are wrong). We have a shopt that tracked daily sales over 4 days (Mon - Thu), and wants to compute for each product how many were sold per day on average. This is the data:

In [ ]:
sales = pd.DataFrame({
    "product": ["cable", "cable", "mouse", "screen", "cable"],
    "day":     ["Mon", "Tue", "Tue", "Wed", "Thu"],
    "units":   [3, 5, 2, 1, 4],
})

display(sales)

So on Monday they sold 3 cables, on Tuesday 4 cables and 2 mice, etc. 

In order to compute daily average sales per product, it might be tempting to groupby `product` and then aggregate with `mean`:

In [ ]:
display(sales.groupby("product")["units"].mean())

But this doesn't produce the correct averages. It does not sell 2 mice per day. It sold two mice on Tuesday and
none for the rest of the week, so an average of 0.5 a mouse per day.

The mean is wrong because `groupby` can only average the rows that exist, but on days no mice were sold, there is no row in the record (the zeros are missing from the data, so they are missing from the average).

We can use `how="cross"`, to create a DataFrame that shows what the complete dataset should look like

In [ ]:
products = pd.DataFrame({"product": ["cable", "mouse", "screen"]})
days     = pd.DataFrame({"day": ["Mon", "Tue", "Wed", "Thu"]})

grid = pd.merge(products, days, how="cross")
display(grid)

That is the scaffold containing every combination of day and product that ought to exist, whether or not anything
happened. Now we can left-join the real sales onto it and fill in the `NaN`-gaps with `0` (since missing data means zero sales).

In [ ]:
# merge scaffold with sales and fill missing values with 0
complete = pd.merge(grid, sales, on=["product", "day"], how="left").fillna(0)
display(complete)

display(complete.groupby("product")["units"].mean())

So now we do have full information that can be used by `groupby` + `mean` to get the average daily sales.

### Why we spread data over multiple tables?

You now know how to combine tables. That raises an obvious question: if `tracks` and
`artists` belong together, why were they ever apart? Why not store one table with
everything in it? Let's create that one big table and look at what problems might arrise:

In [ ]:
wide = pd.merge(tracks, artists, on="artist_id", how="left")

display(wide)

Look at the `country` column. `Neon Harbour` is a Dutch act, and that single fact is now
written on every one of its tracks. Here that is only twice and might not seem like a big deal, but on a real platform, an artist
with 300 tracks would have their country written 300 times.

There is a few issues with that. One is that this might leed to inconsistencies, because a fact stored
in 300 places can disagree with itself, and a fact stored only in one place cannot.

In [ ]:
# Someone fixes a row by hand and leaves a trailing space.
wide.loc[1, "country"] = "NL "

print("countries in the data:", wide["country"].dropna().unique().tolist())
display(wide.groupby("name")["country"].nunique())

With a small faulty edit we now have inconsistent data: `Neon Harbour` is now from two countries. Group by country and you
get four groups where there are three. 

This is one of the reasons we insist so much in Chapter 10 on validating data. But preventing the errors is better than finding them. By having the data split into two tables (`tracks` and `artists`) we have only a single location where the country of an artist is stored. So that leaves no room for inconsistencies.

The same argument applies to changes. If an artist signs to a new label, the two-table
version is a single edit. If we would have one big table, we would need to change this inforamtion on every track that the artist published. With 300 tracks this would become 300 edits, and if you miss one the table now contradicts itself.

There is a third cost, and it is already visible in the data above.

In [ ]:
print("artists in the wide table:", sorted(wide["name"].dropna().unique()))
print("artists we actually have: ", sorted(artists["name"]))

`Odd Even` has vanished. Not because anything went wrong, but because the wide table is a
list of *tracks*, and an artist with no tracks has no row to live on. In one table you
cannot record an artist until they release something — and if you delete an artist's last
track, you lose their country along with it. So the wide table cannot represent things the two narrow tables can.

#### Memory usage

The above focussed on maintaining concistent data. There is another reason to keep data split into multiple tables: We often want to combine tables of very different dimensions. For example a wide and short tabvle (a handful of artists, but a lot
known about each one) and a narrow and tall table (a schedule with a row per day, but little info about the day). 

When you merge them you get the worst of both: every one of those many rows combined with every one
of those many columns.

So here is an example with `artists_full` table, contaning only yhe 4 artists from before, but much more information about each artist (4 rows and 7 columns). And a play schedule, listing when the artists are playing, conatining very little info, but a lot of days (50000 rows and 3 columns).

In [ ]:
import io

# wide and short: a lot known about each of four artists
artists_full = artists.copy()
artists_full["city"]    = ["Rotterdam", "Ghent", "Lyon", "Utrecht"]
artists_full["label"]   = ["Tidal", "Northwind", "Halcyon", "Tidal"]
artists_full["founded"] = [2011, 2016, 2009, 2019]
artists_full["bio"]     = [
    f"Formed in {city}, {name} have released four albums and toured across Europe, "
    "building a reputation for restrained, atmospheric live shows."
    for city, name in zip(artists_full["city"], artists_full["name"])
]

# narrow and tall: one row per artist per day
plays = pd.DataFrame({
    "artist_id": np.tile([1, 2, 3, 4], 12_500),
    "day":       np.arange(50_000) % 365,
    "plays":     np.arange(50_000) % 900,
})

print("artists_full:", artists_full.shape)
print("plays:       ", plays.shape)
display(artists_full)
display(plays)

When we combine them we get a huge table (50000 rows and 9 columns):

In [ ]:
combined = pd.merge(plays, artists_full, on="artist_id")

print("combined:", combined.shape)
display(combined)

Every row now carries a full biography, and there are only four different biographies in the entire table, each one written out 12 500 times. This makes a dramatic difference when we save the files to disk:

In [ ]:
def csv_size_mb(df):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    return len(buffer.getvalue()) / 1_000_000

separate = csv_size_mb(plays) + csv_size_mb(artists_full)

print(f"plays + artists_full: {separate:6.2f} MB")
print(f"combined:             {csv_size_mb(combined):6.2f} MB")
print(f"factor:               {csv_size_mb(combined) / separate:6.1f}x")

The amount oof diskspace that the file occupies is around twenty times more, even though it contains the exact same information. With very large datasets this can become a very big issue.

### Exercise 26

Now do it in reverse. Below is one wide table from a webshop, with the customer's details
repeated on every order they placed. What would be the best way to split up these two tables?

Make the split, so create two tables in such a way that you avoid storing duplicated information.

After creating the two tables, merge them back into one and check that you get the same table as before!

Tip: How to split? You can create smaller tables from the wide tables by using a combination of colum selection (`orders_wide[["col1", "col2", ...]`), `.drop_duplicates()`, and `rest_index(drop=True)`.

In [ ]:
orders_wide = pd.DataFrame({
    "order_id":      [1, 2, 3, 4, 5, 6],
    "customer_id":   [101, 102, 101, 103, 102, 101],
    "customer_name": ["Ada", "Ben", "Ada", "Cleo", "Ben", "Ada"],
    "customer_city": ["Delft", "Ghent", "Delft", "Lyon", "Ghent", "Delft"],
    "product":       ["cable", "mouse", "screen", "cable", "keyboard", "mouse"],
    "amount":        [12.50, 25.00, 189.00, 12.50, 45.00, 25.00],
})

display(orders_wide)

# YOUR CODE HERE